In [3]:
# ====================================================
# INSTALL LIBRARIES
# ====================================================

!pip install ta
!pip install yfinance
!pip install great_expectations
!pip install scikit-learn
!pip install matplotlib
!pip install joblib

  Preparing metadata (setup.py) ... done
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29412 sha256=8c0904082b3cb40610c1f70d7c0ece93ab96c8914a599182c1cb2604ea637797
  Stored in directory: /root/.cache/pip/wheels/5c/a1/5f/c6b85a7d9452057be4ce68a8e45d77ba34234a6d46581777c6
Successfully built ta
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 9.8 MB/s eta 0:00:00


In [11]:
import great_expectations as gx
from great_expectations.validator.validator import Validator
from great_expectations.execution_engine import PandasExecutionEngine
from great_expectations.core.batch import Batch
from great_expectations.expectations.expectation import ExpectationConfiguration
import pandas as pd # Import pandas

print("\n RUNNING GREAT EXPECTATIONS VALIDATION...")

# Initialize Great Expectations DataContext
context = gx.get_context()

# Flatten MultiIndex columns if they exist
# This assumes 'Date' and other relevant column names are at the first level (level 0)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

# Create Batch
batch = Batch(data=df)

# Create Validator
validator = Validator(
    execution_engine=PandasExecutionEngine(),
    batches=[batch]
)

# ====================================================
# EXPECTATIONS
# ====================================================

# Date should not be null
validator.expect_column_values_to_not_be_null("Date")

# Close price validation
validator.expect_column_values_to_be_between(
    "Close",
    min_value=1,
    max_value=100000
)

# Volume should not be negative
validator.expect_column_values_to_be_between(
    "Volume",
    min_value=0
)

# Open should not be null
validator.expect_column_values_to_not_be_null("Open")

# High should be greater than Low
validator.expect_column_pair_values_A_to_be_greater_than_B(
    "High",
    "Low"
)

# Date should be unique
validator.expect_column_values_to_be_unique("Date")

# ====================================================
# VALIDATE
# ====================================================

results = validator.validate()

print("\n VALIDATION RESULTS")
print(results)

INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmppxgfd7z7' for ephemeral docs site
  warnings.warn(




 RUNNING GREAT EXPECTATIONS VALIDATION...


Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

  warnings.warn(



Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

...

Calculating Metrics:   0%|          | 0/7 [00:00<?, ?it/s]

  warnings.warn(



Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/26 [00:00<?, ?it/s]


 VALIDATION RESULTS
{
  "success": false,
  "results": [
    {
      "success": true,
      "expectation_config": {
        "type": "expect_column_values_to_not_be_null",
        "kwargs": {
          "column": "Date"
        },
        "meta": {},
        "severity": "critical"
      },
      "result": {
        "element_count": 740,
        "unexpected_count": 0,
        "unexpected_percent": 0.0,
        "partial_unexpected_list": []
      },
      "meta": {},
      "exception_info": {
        "raised_exception": false,
        "exception_traceback": null,
        "exception_message": null
      }
    },
    {
      "success": true,
      "expectation_config": {
        "type": "expect_column_values_to_be_unique",
        "kwargs": {
          "column": "Date"
        },
        "meta": {},
        "severity": "critical"
      },
      "result": {
        "element_count": 740,
        "unexpected_count": 0,
        "unexpected_percent": 0.0,
        "partial_unexpected_list": [],
 

In [16]:
# The ValidationResultsPageRenderer is typically used for rendering data docs (HTML).
# For extracting data into a DataFrame, we can directly process the results object.

# Extract relevant information to a pandas DataFrame
validation_summary = []
for expectation_result in results.results:
    exp_cfg = expectation_result.expectation_config
    success = expectation_result.success
    result_details = expectation_result.result

    validation_summary.append({
        'Expectation Type': exp_cfg.type,
        'Column': exp_cfg.kwargs.get('column', 'N/A'),
        'Success': success,
        'Element Count': result_details.get('element_count', 'N/A'),
        'Unexpected Count': result_details.get('unexpected_count', 'N/A'),
        'Unexpected Percent': result_details.get('unexpected_percent', 'N/A'),
        'Min Value': exp_cfg.kwargs.get('min_value', 'N/A'),
        'Max Value': exp_cfg.kwargs.get('max_value', 'N/A')
    })

df_results = pd.DataFrame(validation_summary)

In [18]:
# Save the DataFrame to an Excel file
excel_file_path = '/validation_results.xlsx'
df_results.to_excel(excel_file_path, index=False)

print(f"Validation results saved to {excel_file_path}")

Validation results saved to /validation_results.xlsx
